In [1]:
from langchain_core.runnables import RunnableConfig
from dotenv import  load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState, StateGraph,START,END

load_dotenv(override = True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 声明状态
class OverAllState(MessagesState):
    output:str

#2. 声明节点
def llm_mode(state:OverAllState) -> OverAllState:
    messages = state["messages"]
    res = model.invoke(messages)
    return {
        "messages":[res]
    }

def output_node(state:OverAllState) -> OverAllState:
    return {
        "output":state["messages"][-1].content
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_node",llm_mode)
builder.add_node("output_node",output_node)
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node","output_node")
builder.add_edge("output_node",END)

#4. 配置检查点存储器
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

#5. 使用的时候必须填写线程ID


config= RunnableConfig(configurable={
        "thread_id":"chapter03-01"
    })
#6. 执行图
graph.invoke({"messages":[HumanMessage("你好,我是老王")]},config=config)


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='811f5817-7a2b-4126-8740-d6b5a5812b35'),
  AIMessage(content='你好，老王！很高兴认识你。有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都在这里。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 8, 'total_tokens': 37, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '021ac1ac-76b7-4c02-9b49-4462b2894121', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1da62-0345-7294-a84d-26a2535dd7f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 29, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})],
 'output': '你好，老王！很高兴认识你。有什么我可以帮你的吗？无论是

In [2]:
graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config)

{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='811f5817-7a2b-4126-8740-d6b5a5812b35'),
  AIMessage(content='你好，老王！很高兴认识你。有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都在这里。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 8, 'total_tokens': 37, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '021ac1ac-76b7-4c02-9b49-4462b2894121', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1da62-0345-7294-a84d-26a2535dd7f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 29, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  HumanMessage(content='你好,我是谁', additio

In [3]:
config1 = {
    "configurable":{
        "thread_id":"chapter03-01xx"
    }
}
graph.invoke({"messages":[HumanMessage("你好,我是谁")]},config=config1)


{'messages': [HumanMessage(content='你好,我是谁', additional_kwargs={}, response_metadata={}, id='971718c6-fffc-432d-8de2-fbaa6491a143'),
  AIMessage(content='你好！😊\n\n从我们的对话来看，我目前还不知道你的具体身份呢——你还没有告诉我你的名字或者任何关于你的信息。\n\n我是DeepSeek，由深度求索公司创造的AI助手。如果你愿意的话，可以告诉我你的名字，或者你想让我怎么称呼你？这样我们的交流会更亲切一些！\n\n有什么我可以帮你的吗？无论是聊天、解答问题、还是协助你完成什么任务，我都很乐意帮忙！🌟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 8, 'total_tokens': 99, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '463df43f-a620-49e9-b948-dab0236806c5', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1da62-6dde-71b1-8c67-1472ec64652e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 91, 'total_